<a href="https://colab.research.google.com/github/felipefe/1---Factory-Pattern/blob/master/linha_regressao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
from numpy import *

class LinearRegression:
  def __init__(self, x, y):
    self.x = x
    self.y = y
    self.__correlation_coeficiente = self.__correlacao()
    self.__inclination = self.__inclinacao()
    self.__intercept = self.__interceptacao()

  def __correlacao(self):
    covariacao = cov(self.x, self.y, bias=True)[0][1]
    var_x = var(self.x)
    var_y = var(self.y)
    return covariacao / sqrt(var_x * var_y)

  def __inclinacao(self):
    stdx = std(self.x)
    stdy = std(self.y)
    return self.__correlation_coeficiente * (stdy / stdx)

  def __interceptacao(self):
    mediax = mean(self.x)
    mediay = mean(self.y)
    return mediay - (self.__inclination * mediax)

  def prever(self, valor):
    return (self.__inclination * valor) + self.__intercept

In [ ]:
import numpy as np

#Valores
linha = { 18: 871,
          23: 1132,
          28: 1242,
          33: 1356,
          43: 1638,
          48: 2130,
          53: 2454,
          58: 3066,
          63: 4090 }

# Extraindo dados do dicionário
x = np.array(list(linha.keys()))
y = np.array(list(linha.values()))

lr = LinearRegression(x, y)
previsao = lr.prever(int(input('valor prever?')))
print(previsao)


x = array([1, 2, 3, 4, 5])
y = array([2, 4, 6, 8, 10])
lr = LinearRegression(x, y)
print(lr.prever(6))
print(lr.prever(int(input('valor prever?'))))

valor prever?45
2257.151891891892
12.0
valor prever?12
24.0


Cálculo do Beta de uma Ação (Análise de Risco e Volatilidade)O Beta ($\beta$) mede a sensibilidade de uma ação em relação ao mercado como um todo (geralmente representado por um índice como o Ibovespa ou o S&P 500).Como funciona: Você coloca no eixo $X$ os retornos diários ou semanais do índice de mercado e no eixo $Y$ os retornos da ação específica. O coeficiente angular ($w_1$) dessa regressão linear é o Beta.Na prática:
Se o Beta for 1.0, a ação se move na mesma intensidade que o mercado.
Se o Beta for > 1.0 (ex: 1.5), a ação é mais volátil que o mercado (amplifica os movimentos).
Se o Beta for < 1.0 (ex: 0.6), a ação é mais defensiva.

In [ ]:
import yfinance as yf
from numpy import *

#minhaAcao =  'BBDC4.SA'
acoes = ['BBDC4.SA', 'SUZB3.SA', 'EGIE3.SA']

for minhaAcao in acoes:
  # 1. Baixar dados reais de preços
  tickers = [minhaAcao, "^BVSP"]
  dados = yf.download(tickers, start="2020-01-01", end="2027-01-01", auto_adjust= True)["Close"]

  # 2. Calcular o retorno percentual diário e limpar nulos
  retornos = dados.pct_change().dropna()

  # X = Retorno do Ibovespa (Mercado) | Y = Retorno da Petrobras (Ação)
  x = array(retornos["^BVSP"])
  y = array(retornos[minhaAcao])

  lr = LinearRegression(x, y)
  # O coeficiente de inclinação (__inclinacao) aqui representa o BETA da ação!
  print('')
  print(f"O Beta da {minhaAcao} calculado pela sua classe é: {lr._LinearRegression__inclination:.4f}")

[*********************100%***********************]  2 of 2 completed



O Beta da BBDC4.SA calculado pela sua classe é: 1.1534


[*********************100%***********************]  2 of 2 completed
[                       0%                       ]


O Beta da SUZB3.SA calculado pela sua classe é: 0.2973


[*********************100%***********************]  2 of 2 completed


O Beta da EGIE3.SA calculado pela sua classe é: 0.5888


In [ ]:
import yfinance as yf
from numpy import *

# 1. Puxar dados de preços de fechamento
tickers = ["ITUB4.SA", "BBDC4.SA"]
dados = yf.download(tickers, start="2019-01-01", end="2027-01-01", auto_adjust=True)["Close"].dropna()

x = array(dados["ITUB4.SA"])
y = array(dados["BBDC4.SA"])

# 2. Instanciar sua classe de Regressão Linear
lr = LinearRegression(x, y)

# 3. Comparar o preço real do Bradesco com o preço teoricamente previsto pelo Itaú
preco_itub_atual = x[-1]
preco_bbdc_real = y[-1]
preco_bbdc_previsto = lr.prever(preco_itub_atual)

# 4. Calcular o resíduo (desvio do modelo)
residuo = preco_bbdc_real - preco_bbdc_previsto

print(f"Preço Real de BBDC4: R$ {preco_bbdc_real:.2f}")
print(f"Preço Previsto pela Regressão: R$ {preco_bbdc_previsto:.2f}")
print(f"Diferença (Resíduo): R$ {residuo:.2f}")


[*********************100%***********************]  2 of 2 completed

Preço Real de BBDC4: R$ 13.86
Preço Previsto pela Regressão: R$ 16.60
Diferença (Resíduo): R$ -2.74
--- ANÁLISE DE PAIRS TRADING ---
Último Resíduo: R$ -2.74
Desvio Padrão Histórico dos Resíduos: R$ 2.16
Z-Score Atual: -1.27

⚖️ NEUTRO: O par está flutuando dentro da faixa normal de variação (entre -2 e +2 desvios). Sem sinal de entrada.



Como transformar isso em um robô profissional? (O Z-Score)
Um resíduo de -2.89 R$$ por si só não diz muita coisa se não soubermos a volatilidade histórica desse desvio. Para saber se essa diferença é um ponto de entrada real ou apenas ruído normal, os quant traders calculam o Z-Score dos resíduos
O Conceito de Z-Score nos Resíduos
O resíduo isolado (ex: R$ -2,89) nos diz apenas o valor absoluto da diferença. O Z-Score padroniza essa diferença dividindo o resíduo pelo desvio padrão histórico da própria série de resíduos.
Z-Score = 0: O par está perfeitamente na linha de equilíbrio histórico.
Z-Score > +2.0: O ativo Y está esticado para cima (muito caro em relação a X). Gatilho clássico: Vender Y / Comprar X.
Z-Score < -2.0: O ativo Y está esticado para baixo (muito barato em relação a X). Gatilho clássico: Comprar Y / Vender X.


In [ ]:
import yfinance as yf
from numpy import *

dicionarioPairTradding = { 'ITUB4.SA': 'BBDC4.SA',
                           'SUZB3.SA': 'KLBN4.SA',
                           'EGIE3.SA': 'TAEE11.SA',
                           }
for chave, valor in dicionarioPairTradding.items():
  # 1. Puxar dados de preços de fechamento
  tickers = [chave, valor]
  dados = yf.download(tickers, start="2023-01-01", end="2027-01-01", auto_adjust=True)["Close"].dropna()

  x = array(dados[chave])
  y = array(dados[valor])

  # 2. Instanciar a regressão
  lr = LinearRegression(x, y)

  # 3. Calcular os preços previstos para TODOS os dias da série histórica
  # Aplicamos a fórmula y = w1 * x + w0 para cada ponto de x
  y_previstos = (lr._LinearRegression__inclination * x) + lr._LinearRegression__intercept

  # 4. Calcular os resíduos históricos (Preço Real - Preço Previsto)
  residuos = y - y_previstos

  # 5. Calcular a média e o desvio padrão dos resíduos
  media_residuos = mean(residuos)
  std_residuos = std(residuos)

  # 6. Calcular o Z-Score do último dia (dia mais recente)
  ultimo_residuo = residuos[-1]
  z_score = (ultimo_residuo - media_residuos) / std_residuos

  # 7. Exibição dos resultados
  print(f"--- ANÁLISE DE PAIRS TRADING ---")
  print(f"Último Resíduo: R$ {ultimo_residuo:.2f}")
  print(f"Desvio Padrão Histórico dos Resíduos: R$ {std_residuos:.2f}")
  print(f"Z-Score Atual: {z_score:.2f}\n")

  # 8. Regra de Decisão Estatística
  if z_score < -2.0:
      print(f"🚨 SINAL DE COMPRA: O Z-Score está abaixo de -2.0. O ativo Y {valor} está estatisticamente subvalorizado em relação a X.")
  elif z_score > 2.0:
      print(f"🚨 SINAL DE VENDA: O Z-Score está acima de +2.0. O ativo Y ({valor}) está estatisticamente sobrevalorizado em relação a X.")
  else:
      print(f"⚖️ NEUTRO: O par está flutuando dentro da faixa normal de variação (entre -2 e +2 desvios). Sem sinal de entrada.")

[*********************100%***********************]  2 of 2 completed


--- ANÁLISE DE PAIRS TRADING ---
Último Resíduo: R$ -2.77
Desvio Padrão Histórico dos Resíduos: R$ 1.20
Z-Score Atual: -2.32

🚨 SINAL DE COMPRA: O Z-Score está abaixo de -2.0. O ativo Y BBDC4.SA está estatisticamente subvalorizado em relação a X.


[*********************100%***********************]  2 of 2 completed


--- ANÁLISE DE PAIRS TRADING ---
Último Resíduo: R$ -0.13
Desvio Padrão Histórico dos Resíduos: R$ 0.22
Z-Score Atual: -0.57

⚖️ NEUTRO: O par está flutuando dentro da faixa normal de variação (entre -2 e +2 desvios). Sem sinal de entrada.


[*********************100%***********************]  2 of 2 completed

--- ANÁLISE DE PAIRS TRADING ---
Último Resíduo: R$ 9.35
Desvio Padrão Histórico dos Resíduos: R$ 3.00
Z-Score Atual: 3.11

🚨 SINAL DE VENDA: O Z-Score está acima de +2.0. O ativo Y (TAEE11.SA) está estatisticamente sobrevalorizado em relação a X.


In [3]:
import yfinance as yf
import pandas as pd
from numpy import *

def scanner_pairs_trading(dicionario_pares, start_date="2023-01-01", end_date="2027-01-01"):
    resultados = []

    print(f"Escaneando {len(dicionario_pares)} pares na B3...\n")

    for x_ticker, y_ticker in dicionario_pares.items():
        try:
            # 1. Puxar dados históricos
            dados = yf.download([x_ticker, y_ticker], start=start_date, end=end_date, auto_adjust=True)["Close"].dropna()

            if dados.empty:
                continue

            x = array(dados[x_ticker])
            y = array(dados[y_ticker])

            # 2. Executar Regressão Linear
            lr = LinearRegression(x, y)

            # 3. Calcular resíduos e Z-Score
            y_previstos = (lr._LinearRegression__inclination * x) + lr._LinearRegression__intercept
            residuos = y - y_previstos

            media_residuos = mean(residuos)
            std_residuos = std(residuos)

            ultimo_residuo = residuos[-1]
            z_score = (ultimo_residuo - media_residuos) / std_residuos

            # Definir o status/sinal
            if z_score < -2.0:
                sinal = "COMPRA Y (Subvalorizado)"
            elif z_score > 2.0:
                sinal = "VENDA Y (Sobrevalorizado)"
            else:
                sinal = "NEUTRO"

            # Armazenar no formato de dicionário para o DataFrame
            resultados.append({
                "Par": f"{x_ticker} / {y_ticker}",
                "Ativo X": x_ticker,
                "Ativo Y": y_ticker,
                "Preço Y (Real)": round(y[-1], 2),
                "Preço Y (Previsto)": round(y_previstos[-1], 2),
                "Resíduo": round(ultimo_residuo, 2),
                "Z-Score": round(z_score, 2),
                "Sinal": sinal
            })

        except Exception as e:
            print(f"Erro ao processar o par {x_ticker}/{y_ticker}: {e}")

    # Transformar lista em DataFrame do Pandas
    df_resultado = pd.DataFrame(resultados)
    return df_resultado


meus_pares = { 'ITUB4.SA': 'BBDC4.SA',
               'SUZB3.SA': 'KLBN4.SA',
               'EGIE3.SA': 'TAEE11.SA',}

# 1. Rodar o scanner geral (janela a partir de 2023)
tabela_completa = scanner_pairs_trading(meus_pares, start_date="2023-01-01", end_date="2027-01-01")

print("=== TABELA COMPLETA DE RESULTADOS ===")
print(tabela_completa.to_string(index=False))

print("\n" + "="*50 + "\n")

# 2. Filtrar apenas os pares com Z-Score absoluto maior que 2.0 (Oportunidades)
oportunidades = tabela_completa[abs(tabela_completa["Z-Score"]) > 2.0]

print("=== FILTRANDO APENAS OPORTUNIDADES (|Z| > 2.0) ===")
if not oportunidades.empty:
    print(oportunidades.to_string(index=False))
else:
    print("Nenhum par atingiu o gatilho de Z-Score > 2.0 no momento.")

Escaneando 3 pares na B3...



[*********************100%***********************]  2 of 2 completed
[*********************100%***********************]  2 of 2 completed
[*********************100%***********************]  2 of 2 completed


=== TABELA COMPLETA DE RESULTADOS ===
                 Par  Ativo X   Ativo Y  Preço Y (Real)  Preço Y (Previsto)  Resíduo  Z-Score                     Sinal
 ITUB4.SA / BBDC4.SA ITUB4.SA  BBDC4.SA           13.54               16.56    -3.02    -2.52  COMPRA Y (Subvalorizado)
 SUZB3.SA / KLBN4.SA SUZB3.SA  KLBN4.SA            3.43                3.58    -0.15    -0.69                    NEUTRO
EGIE3.SA / TAEE11.SA EGIE3.SA TAEE11.SA           39.25               30.19     9.06     3.00 VENDA Y (Sobrevalorizado)


=== FILTRANDO APENAS OPORTUNIDADES (|Z| > 2.0) ===
                 Par  Ativo X   Ativo Y  Preço Y (Real)  Preço Y (Previsto)  Resíduo  Z-Score                     Sinal
 ITUB4.SA / BBDC4.SA ITUB4.SA  BBDC4.SA           13.54               16.56    -3.02    -2.52  COMPRA Y (Subvalorizado)
EGIE3.SA / TAEE11.SA EGIE3.SA TAEE11.SA           39.25               30.19     9.06     3.00 VENDA Y (Sobrevalorizado)
